# 10 - Export Hybrid Ternary MLP Package

Exporta um pacote fiel ao resultado `l78`: MLPs ternários empacotados em 2-bit + adapter LoRA de recovery. O ternário puro sem recovery é um modelo morto; o artefato correto é híbrido.

In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, math, gc

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    !pip -q install transformers accelerate sentencepiece safetensors peft
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
EXPORT_DIR = PROJECT_ROOT / 'exports' / 'ternary_mlp_l78'
for p in [RESULTS_DIR, CHECKPOINTS_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('EXPORT_DIR =', EXPORT_DIR)


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import AutoPeftModelForCausalLM

HF_TOKEN = os.environ.get('HF_TOKEN')
PRUNED_CHECKPOINT = CHECKPOINTS_DIR / 'student_pruned'
L78_ADAPTER = CHECKPOINTS_DIR / 'student_ternary_recovered_l78'

# Export choices.
# We export the ternary MLP base and keep the l78 recovery adapter as part of the package.
LOAD_RECOVERED_ADAPTER = False
MAX_TENSORS_TO_EXPORT = None  # None = all MLP tensors. Use small int for debugging.
PACK_VALUES = True
APPLY_TERNARY_SNAP_BEFORE_EXPORT = True
INCLUDE_RECOVERY_ADAPTER_IN_PACKAGE = True

print('PRUNED_CHECKPOINT exists:', PRUNED_CHECKPOINT.exists(), PRUNED_CHECKPOINT)
print('L78_ADAPTER exists:', L78_ADAPTER.exists(), L78_ADAPTER)


In [ ]:
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def is_mlp_projection_weight(name):
    return (
        '.mlp.' in name
        and name.endswith('.weight')
        and any(part in name for part in ['gate_proj', 'up_proj', 'down_proj'])
    )


def iter_mlp_projection_weights(model):
    # named_parameters is robust for both base and PEFT/wrapped models.
    for name, param in model.named_parameters():
        if is_mlp_projection_weight(name):
            yield name, param.detach()


def ternary_codes_from_weight(weight):
    # Recompute nearest ternary code from recovered weights.
    # code: 0 -> zero, 1 -> +scale, 2 -> -scale, 3 reserved.
    w = weight.detach().float().cpu()
    scale = w.abs().mean().clamp(min=1e-8)
    q = torch.clamp(torch.round(w / scale), -1, 1).to(torch.int8)
    codes = torch.zeros(q.numel(), dtype=torch.uint8)
    flat = q.flatten()
    codes[flat > 0] = 1
    codes[flat < 0] = 2
    return codes, float(scale), tuple(w.shape)


def pack_2bit(codes):
    codes = codes.to(torch.uint8).flatten()
    n = codes.numel()
    pad = (-n) % 4
    if pad:
        codes = torch.cat([codes, torch.zeros(pad, dtype=torch.uint8)])
    codes = codes.view(-1, 4)
    packed = codes[:, 0] | (codes[:, 1] << 2) | (codes[:, 2] << 4) | (codes[:, 3] << 6)
    return packed.numpy().tobytes(), n, pad


def unpack_2bit(blob, n, shape, scale):
    raw = torch.tensor(list(blob), dtype=torch.uint8)
    codes = torch.empty(raw.numel() * 4, dtype=torch.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    codes = codes[:n]
    vals = torch.zeros(n, dtype=torch.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    return vals.view(shape)


def human_bytes(n):
    units = ['B','KB','MB','GB','TB']
    v = float(n)
    for u in units:
        if v < 1024 or u == units[-1]:
            return f'{v:.2f} {u}'
        v /= 1024


def dir_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file())


In [ ]:
cleanup_memory()

if LOAD_RECOVERED_ADAPTER:
    print('Loading recovered l78 adapter model...')
    model = AutoPeftModelForCausalLM.from_pretrained(
        str(L78_ADAPTER),
        token=HF_TOKEN,
        device_map='cpu',
        low_cpu_mem_usage=True,
    )
else:
    print('Loading pruned checkpoint...')
    model = AutoModelForCausalLM.from_pretrained(
        str(PRUNED_CHECKPOINT),
        token=HF_TOKEN,
        device_map='cpu',
        low_cpu_mem_usage=True,
    )
model.eval()
mlp_names = [name for name, _ in iter_mlp_projection_weights(model)]
print('loaded')
print('MLP projection tensors found:', len(mlp_names))
print('First tensors:', mlp_names[:6])
if not mlp_names:
    raise RuntimeError('No MLP projection tensors found. Check model parameter naming.')


In [ ]:
manifest = {
    'format': 'rpt_hybrid_ternary_mlp_lora_v1',
    'codebook': {'0': '0', '1': '+scale', '2': '-scale', '3': 'reserved'},
    'scale': 'per tensor mean(abs(weight)) recomputed at export after optional snap',
    'source_adapter': str(L78_ADAPTER),
    'include_recovery_adapter_in_package': INCLUDE_RECOVERY_ADAPTER_IN_PACKAGE,
    'adapter_size_bytes': dir_size_bytes(L78_ADAPTER),
    'adapter_size_human': human_bytes(dir_size_bytes(L78_ADAPTER)),
    'apply_ternary_snap_before_export': APPLY_TERNARY_SNAP_BEFORE_EXPORT,
    'source_pruned_checkpoint': str(PRUNED_CHECKPOINT),
    'tensors': [],
}

bin_path = EXPORT_DIR / 'mlp_ternary_2bit.bin'
if bin_path.exists():
    bin_path.unlink()

fp16_total_bytes = 0
packed_total_bytes = 0
num_tensors = 0
max_abs_reconstruction_error = 0.0

with open(bin_path, 'wb') as bf:
    for name, weight in iter_mlp_projection_weights(model):
        if MAX_TENSORS_TO_EXPORT is not None and num_tensors >= MAX_TENSORS_TO_EXPORT:
            break
        # Export the ternary snapped representation. This creates the packed artifact;
        # recovery quality was already validated by the l78 experiment.
        codes, scale, shape = ternary_codes_from_weight(weight)
        blob, n, pad = pack_2bit(codes)
        offset = bf.tell()
        bf.write(blob)
        size = len(blob)
        packed_total_bytes += size
        fp16_total_bytes += n * 2
        num_tensors += 1

        # Reconstruction test on exported codes against ternary snapped version, not original fp recovered weights.
        recon = unpack_2bit(blob, n, shape, scale)
        q = torch.clamp(torch.round(weight.detach().float().cpu() / scale), -1, 1) * scale
        err = float((recon - q).abs().max().item())
        max_abs_reconstruction_error = max(max_abs_reconstruction_error, err)

        manifest['tensors'].append({
            'name': name,
            'shape': list(shape),
            'numel': int(n),
            'scale': scale,
            'offset': offset,
            'packed_bytes': size,
            'pad_codes': int(pad),
            'fp16_bytes': int(n * 2),
            'compression_vs_fp16': (n * 2) / size if size else None,
            'max_abs_reconstruction_error': err,
        })

manifest['summary'] = {
    'package_type': 'packed ternary MLP base + LoRA recovery adapter',
    'num_tensors': num_tensors,
    'fp16_total_bytes': fp16_total_bytes,
    'fp16_total_human': human_bytes(fp16_total_bytes),
    'packed_total_bytes': packed_total_bytes,
    'packed_total_human': human_bytes(packed_total_bytes),
    'compression_vs_fp16': fp16_total_bytes / packed_total_bytes if packed_total_bytes else None,
    'max_abs_reconstruction_error': max_abs_reconstruction_error,
    'adapter_size_bytes': dir_size_bytes(L78_ADAPTER),
    'adapter_size_human': human_bytes(dir_size_bytes(L78_ADAPTER)),
    'hybrid_package_bytes_excluding_base_non_mlp': packed_total_bytes + dir_size_bytes(L78_ADAPTER),
    'hybrid_package_human_excluding_base_non_mlp': human_bytes(packed_total_bytes + dir_size_bytes(L78_ADAPTER)),
}
if num_tensors != 78:
    raise RuntimeError(f'Expected 78 MLP tensors, exported {num_tensors}. Refusing to accept incomplete package.')

manifest_path = EXPORT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest['summary'], ensure_ascii=False, indent=2))
print('BIN:', bin_path)
print('MANIFEST:', manifest_path)


In [ ]:
# Save a compact report into results.
report = {
    'export_dir': str(EXPORT_DIR),
    'manifest_path': str(EXPORT_DIR / 'manifest.json'),
    'bin_path': str(EXPORT_DIR / 'mlp_ternary_2bit.bin'),
    'summary': manifest['summary'],
    'important_note': 'The functional l78 result requires the recovery adapter. Packed ternary MLP alone corresponds to the pre-recovery collapsed model.',
    'first_tensors': manifest['tensors'][:5],
}
report_path = RESULTS_DIR / 'deployment_audit' / 'ternary_mlp_export_report.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('REPORT:', report_path)
print(json.dumps(report, ensure_ascii=False, indent=2)[:4000])


## Próximo passo

Se o export gerar `78` tensores e `~395 MB` para os MLPs, o pacote correto é `mlp_ternary_2bit.bin + manifest.json + student_ternary_recovered_l78 adapter`. O próximo notebook deve validar o carregamento desse pacote híbrido.